# Exp17 — Left-Relation MLP Ablation (Qwen2-VL-2B)
Tests whether layers 21 & 23 are causal for **left** spatial reasoning too.
If yes → circuit is relation-general, not a vflip artifact.

In [ ]:
# 1. Install dependencies
!pip install transformers accelerate qwen-vl-utils -q

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# 2. Clone repo
import os
if not os.path.exists('Mechanistic-interpretability-research'):
    !git clone https://github.com/likithp82/Mechanistic-interpretability-research.git
%cd Mechanistic-interpretability-research
!git log --oneline -3

In [ ]:
# 3. Download COCO val2017 images + manifest
# The manifest.jsonl is in the repo already.
# We only need the actual images — download from COCO.
import os
os.makedirs('src/dataset/spatial_real_dataset_val2017/images', exist_ok=True)

if not os.path.exists('/tmp/val2017.zip'):
    !wget -q http://images.cocodataset.org/zips/val2017.zip -O /tmp/val2017.zip
    !unzip -q /tmp/val2017.zip -d /tmp/

# Copy only the images referenced in the manifest
import json, shutil
needed = set()
with open('src/dataset/spatial_real_dataset_val2017/manifest.jsonl') as f:
    for line in f:
        row = json.loads(line)
        needed.add(row['output_file'].split('/')[-1])

dest = 'src/dataset/spatial_real_dataset_val2017/images'
copied = 0
for fname in needed:
    src_path = f'/tmp/val2017/{fname}'
    if os.path.exists(src_path):
        shutil.copy2(src_path, f'{dest}/{fname}')
        copied += 1
print(f'Copied {copied}/{len(needed)} images')

In [ ]:
# 4. Run Exp17 — MLP-only sweep for 'left' relation, layers 18-24
# ~17 min on T4 (MLP phase only)
!PYTHONUNBUFFERED=1 python -u src/day7/exp15_attention_head_sweep.py \
    --manifest src/dataset/spatial_real_dataset_val2017/manifest.jsonl \
    --images-root src/dataset/spatial_real_dataset_val2017/images \
    --max-images 60 \
    --layer-window 3 \
    --ablate-mlp \
    --relation left \
    --out-dir src/day7 \
    2>&1 | tee src/day7/exp17_left_mlp_run.log

In [ ]:
# 5. Print results summary
import json
with open('src/day7/results_exp15.json') as f:
    r = json.load(f)

print(f"Relation: {r.get('relation', 'above')}")
print(f"Question: {r['fixed_question']}")
print()
print('MLP flip rates by layer:')
for layer, fr in sorted(r['mlp_flip_rates'].items(), key=lambda x: int(x[0])):
    bar = '█' * int(float(fr) * 30)
    print(f'  Layer {int(layer):2d}: {float(fr):.4f}  {bar}')